## Project 1 - Linear Regression as a Classification Model

#### Objective
Using real world data from actual users of nike.com, determine if: a page view (visit) for a product page was viewed on a desktop platform or a Mobile device.

The following four performance based metrics are used to train the supervised model:

1. `Dom Processing Duration` - (seconds) time for the web browser to build the Document Object Model (typically when the page is ready to interact with )
2. `Page Rendering Duration` - (seconds) time after dom processing is complete and content from the dom is rendered
3. `Network Duration` - (seconds) time spent receiving the initial document, considerably slower on slower connections, ie Mobile 3g
4. `First Contentful Paint` - (seconds) time when the user first sees something on the page, this metric is sensitive to the above 3 metrics

Giving typical industry standards, it's known that Desktop platforms have more available: cpu processing power, memory and usually internet bandwidth when compared to Mobile devices. These metrics provide a strong role in describing the user's topology.  However, there is variance, some Mobile devices are powerful enough to appear as a Desktop device, corollary, Desktop platforms may have a slow connection, low cpu processing power and appear to perform closer to a Mobile device.

Let's keep this assumption in mind through this statistical analysis.

To keep the dataset as clean as possible, the data has been scoped to within the US.  Also, all data was retrieved with the same browser type across Desktop and Mobile - which is the `Safari` browser.

In addition the nike.com page is a product display page of a popular product (url not listed here to keep data anonymous).  Using the same page helps to reduce variance amongst user visits.

#### Analysis Plan 
- Split the model into a training set and testing set (80/20)

- Use One Hot Encoding to classify the output:
    - Desktop: [1,0]
    - Mobile: [0,1]

- Fit the linear regression model on the supervised training set
- Provide analysis of results with supporting charts, mesh grid, confusion matrix, OLS, MSE
- Fit a logistic regression model
- Provide analysis of how the regression model compares to the linear model
- Provide overall summary and next steps

The main objective is: use data that shows no obvious outcomes (Desktop or Mobile) when viewed with the human eye. For example, attributes (features) such as: screen size, orientation, or OS name would provide a direct relationship to the outcome (dependent variable). Here, we must rely on less obvious attributes and how the those metrics work together to give strong signals towards the dependent variable.

Another point is multicollinearity and how that affects our prediction model.  The metrics available can have a dependence on each other as they could typically happen in a linear order.  I don't think this is a problem for a prediction model.  Reason being, we are concerned with how all the metrics work together to predict an outcome vs how 1 metric in isolation can affect an outcome by a certain degree.




#### Data Retrieval

Data was pulled from a 3rd party vendor (New Relic) that is used to track user visits with their associated performance metrics and meta data.
It was queried with the New Relics query UI and exported as csv files:
1. 5000Desktop.csv
2. 5000Mobile.csv

With the amount of traffic on nike.com the query timeline was less than 1 day each of Desktop and Mobile.

On initial data retrieval, fields with null values were excluded.

5000 records is a large dataset for this prediction model, though with the amount of variance or overlap in data, my thinking is: a large data set will improve the model's stability and potentially the prediction accuracy. 

Though, this would not hold true if the overall signals follow the same patterns as a smaller data set.

It would be interesting to run this model with less data to determine the minimum set of data that provides the same accuracy.


In [8]:
# setup standard imports for data analysis and visualization
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns   
import statsmodels.api as sm

# modeling and evaluation
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, classification_report
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error

#### Data Preprocessing

- records were scoped to the US country region
- records were scoped to the Safari browser
- some records from the source had null values for at least one feature, we will filter out those records here
- it was noticed in the scatter plot that outliers were causing the almost all the data to be 'bunched' into the lower corner, we will remove outliers here to prevent that at the 99th percentile

In [9]:

# drop any row that has at least one missing value
# note: the initial query from New Relic should have dealt with this
def clean_data(df):
    # drop rows with missing values
    initial_row_count = len(df)
    df = df.dropna()
    post_row_count = len(df)
    print(("preprocessing: removed {} rows with missing values").format(initial_row_count - post_row_count))
    
    return df


# on viewing of the scatter plot initially, we saw outliers that made all the bulk of the data compressed in the lower left corner
# removing outlier above the 99th percentile for each feature, this is a common method to remove outliers in a dataset
def remove_outliers(df):
    rows_before = len(df)
    for col in featureList:
        upper_bound = df[col].quantile(0.99)
        df = df[df[col] <= upper_bound]
    print(f"Dropped {rows_before - len(df)} rows above the 99th percentile cutoffs.")
    return df

In [10]:
df_mobile = pd.read_csv("5000Mobile.csv")
df_desktop = pd.read_csv("5000Desktop.csv")

# load data in data frames: read each csv file and combine into a single dataframe
df = pd.concat([df_mobile, df_desktop], ignore_index=True)

# from the csv header, set the feature list and target variable
featureList = ["Dom Processing Duration", "Page Rendering Duration", "Network Duration", "First Contentful Paint"]
target = "Device Type"

# preprocess data, remove any rows with missing values, and handle outliers
df = clean_data(df)
df = remove_outliers(df)



FileNotFoundError: [Errno 2] No such file or directory: '5000Mobile.csv'

#### One Hot Encode Output Labels

- create one-hot encoding using np.eye
- this will encode all the y values results in a roughly 10,000 x 2 matrix (less rows removed from 99th percentile outliers)

In [ ]:


unique_values, indices = np.unique(df[target], return_inverse=True)
Y_ohe = np.eye(len(unique_values))[indices]
print("one-hot encoded labels shape: {}".format(Y_ohe.shape))
print(Y_ohe[:1]) 



#### Training and Testing Data

- Split the dataframe into a training set and a testing set
- we need some additional lists here:
    - X_train, X_test, Y_train, Y_test: are lists to feed the models
    - y_train_idx, y_test_idx: are lists for sanity checks and for the upcoming confusion matrix

In [ ]:

# will assign 0 to Desktop and 1 to Mobile, this is the order of the unique values in the target column
class_names = sorted(df[target].unique()) 

class_to_idx = {name: idx for idx, name in enumerate(class_names)}

# the output for each row, either a '0' or '1'
y_indices = np.array([class_to_idx[label] for label in df[target]])

# split the data into training and testing sets
# for each list we pass in, we get back two lists: one for training and one for testing
# 20% of the data will be used for testing, and the rest for training
# 42 is the random seed for reproducibility (42 is typically used)
# stratify based on class labels to ensure an even split
X_train, X_test, y_train, y_test, y_train_idx, y_test_idx = train_test_split(df[featureList], Y_ohe, y_indices, test_size=0.2, random_state=42, stratify=y_indices)


print(f"Training set size: {len(X_train)}")
print(f"Testing set size:  {len(X_test)}")

#### Scatterplot Visualization

- The next step is to plot 2 features in a scatterplot, but the question is which 2 features from the 4 features do we pick?

- There is a concept named **"Fisher's Score"** where we find the difference in the means squared divided by the sum of the variances.

- Use this method to find the 2 features that rank the highest, giving a strong ability to distinguish between classes - this method is good for supervised feature selection.



In [ ]:
def get_best_two_features(X, y):
    f_ratios = []
    for i in range(X.shape[1]):  # Loop through all 4 features
        feature_data = X[:, i]
        # Separate the feature into the two classes (0 and 1), ie all the values of the feature for class 0 or 1
        class0 = feature_data[y == 0] 
        class1 = feature_data[y == 1]
        
        # Math: (Difference in Means)^2 / (Sum of Variances)
        numerator = (np.mean(class0) - np.mean(class1))**2
        denominator = np.var(class0) + np.var(class1)
        f_ratios.append(numerator / denominator)
    
    # Return the indices of the 2 features with the highest scores
    return np.argsort(f_ratios)[-2:]

# Usage
best_pair = get_best_two_features(X_train.values, y_train_idx)
print("feature names: {}".format([featureList[i] for i in best_pair]))

Now that we have our 2 best features, let's plot them on a scatter plot and see how the separation looks.

Since the model has 4 input features, we visualize only 2 at a time using a 2D scatter plot.

In [ ]:
x_axis_feature = "Page Rendering Duration"
y_axis_feature = "First Contentful Paint"

plt.figure(figsize=(9, 6))
plt.scatter(
    X_train[x_axis_feature],
    X_train[y_axis_feature],
    c=y_train_idx,
    edgecolors='k',
    cmap=plt.cm.jet,
    alpha=0.6
)
plt.title("Training Set by Class (red=mobile, blue=desktop)")
plt.xlabel(x_axis_feature)
plt.ylabel(y_axis_feature)
plt.show()

The scatter plot shows substantial overlap, especially in the lower-left region, which suggests that when `First Contentful Paint` and `Page Rendering Duration` have low values, the classes are harder to distinguish. 

Later will we create the confusion matrix - it should show a high number of misclassifications between the 2 classes.

#### Fit Linear Regression Model with the One Hot Encoding Output

In [ ]:
# Create and fit the linear regression model
model = LinearRegression()
model.fit(X_train, y_train)

Y_pred_score = model.predict(X_test) # find the score for each input row of the test set

# for each row, find the index of the highest score, this will be the predicted class (0 or 1) 
# 0 = Desktop and 1 = Mobile
y_pred_idx = np.argmax(Y_pred_score, axis=1) 
print(y_pred_idx[:5])


#### Calculate Accuracy

- We have an accuracy of about **69%** which is above our target of plus **60%**
- Given the fact we are using a linear regression model, which typically is not used for classification, this is a pretty decent result.

In [ ]:
# given the true labels and predicted labels, we can calculate the accuracy of the model

accuracy = accuracy_score(y_test_idx, y_pred_idx)
print(f"Accuracy of the model: {accuracy:.4f}")



#### Approach for the Analysis of Linear Regression as a Classifier Model

We'll use the following as supporting visualizations and predictor effects
1) Mesh Grid
2) Confusion matrix
3) OLS summary 
4) MSE

In [ ]:
# Setting up a meshgrid for the decision boundary
x1_min, x1_max = df[x_axis_feature].min() - 0.2, df[x_axis_feature].max() + 0.2
x2_min, x2_max = df[y_axis_feature].min() - 0.2, df[y_axis_feature].max() + 0.2
xx1, xx2 = np.meshgrid(np.linspace(x1_min, x1_max, 300),np.linspace(x2_min, x2_max, 300))

# we can only visualize 2 features at a time, fixing the other two features to their median values seems reasonable
# median values remove the influence of outliers, we don't want to remove them completely, as they are part of the data and the decision boundary should reflect them as well
# initially we hold all 4 features at their median values
# we'll use the training set to calculate the median values, to keep the test set isolated

median_values = X_train.median()
grid_df = pd.DataFrame({col: np.full(xx1.size, median_values[col]) for col in featureList})


# now we update the two features we are plotting with the values from the meshgrid,
# this will allow us to get the predicted scores for each point in the meshgrid,
# which we can then use to plot the decision boundary
# we use ravel to flatten the meshgrid arrays into 1D, this is needed for the prediction step
grid_df[x_axis_feature] = xx1.ravel()
grid_df[y_axis_feature] = xx2.ravel()

grid_pred_scores = model.predict(grid_df[featureList])
grid_pred_idx = np.argmax(grid_pred_scores, axis=1)
grid_pred_idx = grid_pred_idx.reshape(xx1.shape)

# Plot decision boundaries and data points
plt.figure(figsize=(9, 6))

plt.contourf(xx1,xx2, grid_pred_idx, alpha=0.35, levels=np.arange(-0.5, len(class_names) + 0.5, 1), cmap=plt.cm.jet, vmin=0, vmax=len(class_names) - 1, )

# plot the testing data points
plt.scatter(X_test[x_axis_feature], X_test[y_axis_feature], c=y_test_idx, edgecolors='k', cmap=plt.cm.jet, vmin=0, vmax=len(class_names) - 1, alpha=0.8 )
plt.title("Classification with One Hot Encoding (red=mobile, blue=desktop)")
plt.xlabel(x_axis_feature)
plt.ylabel(y_axis_feature)
plt.show()


#### Analysis of the Mesh Grid and Classification 

The mesh grid shows the scatterplot of the testing data.  We can see that, generally, it is the same distribution as the training data scatterplot, which is what we'd expect.

The boundary line being at an angle confirms what we already have hypothesized, that the prediction depends on a combination of both features.

We can see that for Mobile, many of the red dots, are above the line, suggesting that Mobile is correctly being classified.

However, we also see many red and blue dots on both sides of the boundary line, indicating that many are misclassified.

Let's check the confusion matrix to validate and bring more clarity to these misclassifications.


#### Confusion Matrix

Compare the actual classes with the predicted classes

In [ ]:

y_pred_labels = np.array([class_names[i] for i in y_pred_idx])
y_test_labels = np.array([class_names[i] for i in y_test_idx])

# Calculate confusion matrix
conf_matrix = pd.crosstab(
    pd.Series(y_test_labels, name='Actual'),
    pd.Series(y_pred_labels, name='Predicted')
)

sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues')
plt.title("Confusion Matrix")
plt.show()



Let's review the Precision and Recall as these are important metrics to use when evaluating a classification model

| Class | Precision | Recall |
|--|--|--|
| Desktop | 874/1369 | 874/978 |
| Mobile | 448/552 | 448/943 |

which equates to:

| Class | Precision | Recall |
|--|--|--|
| Desktop | 0.6384 | 0.8937 |
| Mobile | 0.8116 | 0.4751 |

Alternatively, we can use the handy `classification_report` function.

In [ ]:
print(classification_report(y_test_labels, y_pred_labels, target_names=class_names))

#### Confusion Matrix Analysis

We can see the precision for Mobile is pretty strong, but the recall is weak, meaning the model missed many actual Mobile cases.

Desktop performed better overall with the higher recall and a decent precision.

The F1 scores reflect this balance and shows the model is more reliable for predicting Desktop.

In conjunction with the mesh grid, we can see that when the model predicts Mobile it is usually correct, with red points being in the upper (red) region, out on their own.  However, there are many red dots that are incorrectly sitting below the boundary line, which brings down the recall significantly.

With the caveat, that the mesh grid is only showing 2 of the 4 features, while the confusion matrix is using all 4 features.  The mesh grid provides a nice visual but it does not show the full picture.

Also, side note: we can see the same accuracy score as generated above.

#### Ordinary Least Squares (OLS) evaluation

It would be interesting to view coefficients values to help determine which features contribute most strongly to a Desktop or Mobile decision.

We'll use the training data to be consistent with the fitted model.

We'll also focus on Desktop, as for Mobile, since this is a 2 class setup, we would expect the coefficients to have the same values but complementary signs.

In [ ]:
X_sm = sm.add_constant(pd.DataFrame(X_train, columns=featureList, index=X_train.index))
focus_class = 'Desktop' if 'Desktop' in class_to_idx else class_names[0]
is_desktop_train = pd.Series(y_train[:, class_to_idx[focus_class]], index=X_train.index)

smodel = sm.OLS(is_desktop_train, X_sm).fit()
print(smodel.summary())

#### Analysis of the OLS Summary

The p-values being 0 for all 4 features suggest all features are statistically relevant for the Desktop linear fit.

The R-squared value says that the linear fit explains a low amount of the variation in the Desktop target.  This isn't particularly negative, as the low p-values suggest the coefficients are statistically significant.

OLS is not the best tool to use for feature selection, it's useful here as a supportive signal. The confusion matrix is better suited for the performance of this model.

#### Linear Regression Mean Squared Error

MSE tells us how close our models predicted values are to the observed values

In [ ]:
mse = mean_squared_error(y_test, Y_pred_score)
print(f"Mean Squared Error: {mse:.4f}")

#### MSE Analysis

A MSE of 0.2143 is in line with the expectation given we saw an accuracy of 69%.  A low MSE is good, and this suggests the model's prediction is fairly close to the one hot encoded labels.  One thing to note, for a classification it is more suitable to focus on: accuracy, confusion matrix and F1 scores.

However, this will be useful to compare to the MSE from the Logistic Regression section.

#### Comparison with Logistic Regression

Given the fact we used one hot encoding, a more suitable model for prediction would be Logistic Regression.  Logistic Regression outputs values between 0 and 1 and is designed for classification decisions.

In [ ]:
# Fit logistic regression and evaluate accuracy on the test set
logistic_scaler = StandardScaler()
X_train_log_scaled = logistic_scaler.fit_transform(X_train)
X_test_log_scaled = logistic_scaler.transform(X_test)

logistic_model = LogisticRegression(max_iter=1000, random_state=42)
logistic_model.fit(X_train_log_scaled, y_train_idx)

logistic_pred_idx = logistic_model.predict(X_test_log_scaled)
logistic_accuracy = accuracy_score(y_test_idx, logistic_pred_idx)

print(f"Logistic regression accuracy: {logistic_accuracy:.4f}")

#### Logistic Regression Mean Squared Error

See how the mean squared error for logistic regression compares to the value for the linear regression

In [ ]:
logistic_pred_proba = logistic_model.predict_proba(X_test_log_scaled)
logistic_mse = mean_squared_error(y_test, logistic_pred_proba)
print(f"Logistic Regression Mean Squared Error: {logistic_mse:.4f}")

#### Logistic Regression Mean Squared Error Analysis

Logistic Regression MSE is 0.2021 vs 0.2143 for Linear Regression

Lower is better, and this value is slightly better vs Linear Regression MSE.

This is inline with our Logistic Regression values (logistic regression accuracy score) as compared with it's respective Linear Regression.

#### Logistic Regression Confusion Matrix

Now that we viewed the accuracy score and MSE, the next step is to see how the confusion matrix compares to the linear regression confusion matrix

In [ ]:
# Logistic regression confusion matrix
logistic_pred_labels = np.array([class_names[i] for i in logistic_pred_idx])
logistic_test_labels = np.array([class_names[i] for i in y_test_idx])

logistic_conf_matrix = pd.crosstab(
    pd.Series(logistic_test_labels, name='Actual'),
    pd.Series(logistic_pred_labels, name='Predicted')
)

sns.heatmap(logistic_conf_matrix, annot=True, fmt='d', cmap='Blues')
plt.title("Logistic Regression Confusion Matrix")
plt.show()

In [ ]:
# Logistic regression classification report
print(classification_report(logistic_test_labels, logistic_pred_labels, target_names=class_names))

##### Linear Regression as a Classifier vs Logistic Regression Analysis


My assumption before running the logistic regression was that it would give an accuracy score notably higher vs linear regression, given the stated fact that logistic regression is well suited for classification.

Regardless, our logistic regression had a very similar accuracy score, 70%, only 1 percentage point higher.  We can say that using linear regression as a classifier model holds up well as it's results are very close to logistic regression.

The confusion matrix also trends similarly in that the Desktop class still had a better overall F1-score vs Mobile.
It did show that Mobile had a lower recall score, suggesting that many Mobile observations are being predicted as Desktop.  I would say logistic regression makes this point stronger in the data.

Overall, I would tend to make further decisions based on the logistic regression analysis, though very similar.  The takeaway here is that the mobile signals need to be stronger to increase the recall score.

### Final Analysis and Summary

##### Challenges Faced

Initially this was 3 class problem with: Desktop, Mobile, Tablet

Tablet introduced too much variance and observations boundaries were not clear between the 3 data sets
accuracy was ~45%.

Since there were so few tablet users compared to Desktop and Mobile, I had to query back in time for 90 days, vs less than 1 day for Desktop and Mobile - a longer time duration added more variance to the data as it included more environmental network changes and potentially changing web page images and html structure.

The low accuracy number lead to the decision to not include tablet and focus the data solely on Desktop and Mobile.

It also lead me to make the sample data for Desktop and Mobile as stable as possible by focusing on a single product page and users within the US.

The next challenge, was to retrieve all records that did not contain any null values, I was able to do this from querying at the source with a 'NOT NULL' clause in the New Relic Query.  This maintained the sample data at 5000 records.

The next challenge, was the issue of outliers.  Here the pandas api was well suited to filter out records, if any one of the 4 features, were in the 99th percentile.

##### Model's Performance

To understand the model performance we'll focus on the confusion matrix and the scatter plot with the mesh grid boundary line.

Overall the linear regression model achieved a 69% accuracy score, ranking this model in the moderate range for classification performance.  It was able to hit the target of above 60% showing there was enough information to determine a user's visit being on Desktop vs Mobile.

The rational for using linear regression as a classifier was more of an exercise in learning.  It provided a linear model and gave us the opportunity to explore OLS as supportive signals.  Overall, it provided a good baseline to analysis the how the well the models prediction scored.  From here, it would be interesting to dig deeper into other statisstical models, variable selection and adding other features with the goal to increase the accuracy. 

Looking at the confusion matrix, we see Desktop has a recall rate of 89% vs 48% for Mobile.  Therefore, it correctly classified most Desktop observations.  However, it incorrectly classified many Mobile observations as Desktop.

On the corollary, for Mobile, precision is higher at 81%, which shows that when a Mobile prediction is made, it's often correct, but fails to predict many observations and classifies them as Desktop.

The model is biased towards predicting Desktop and the higher F1 score for Desktop reflects this at 74% Desktop vs 60% Mobile

This performance shows up visually on the scatter plot and mesh grid boundary line.  We can see a lot of overlap between the classes, and the linear boundary does not cleanly separate them.


##### Next Steps and Thoughts

Notable takeaways:

- Firstly, we need another metric that has a strong signal to bring the model accuracy up, particullarly for Mobile.  There is one more metric that was not included here due to a higher degree of complication in the data gathering phase.  That metric is Cumulative Layout Shift (CLS).  It measures the shiftiness of the page as it renders (continuous value: 0-100), and it looks to be significantly different (on average) per device.  It may even be significant enough to bring back in the 3rd device type: Tablet.

- Secondly, running this analysis with much smaller sample sizes to determine the minimum sample size, that gives the same accuracy number, would be beneficial in terms of maximizing overall model efficiency.


